# SENTINEL-GNSS — Kaggle Training Notebook

**Before running:** Settings ▸ Accelerator ▸ **GPU T4 x2** (or P100)  
**Also required:** Settings ▸ **Internet on** (needed to clone from GitHub)

| What lives where | |
|---|---|
| Code + all data CSVs | GitHub (cloned in Step 1) |
| Feature windows (.npz) | Built fresh in `/kaggle/working/sentinel-gnss/` |
| Checkpoints + figures | `/kaggle/working/sentinel-gnss/results/` (download via Output tab) |

> **Differences from Colab version:** No Google Drive mount needed.  
> All outputs are written to `/kaggle/working/sentinel-gnss/results/` and are  
> available for download from the notebook's **Output** tab after the run completes.  
> Kaggle sessions are longer-lived than Colab (up to 12 h), but if a session is  
> interrupted you will need to re-run from Step 1.

## Step 1 — Clone repo from GitHub
All code and processed CSVs (including the labelled dataset) are pulled directly from GitHub.  
**Internet must be enabled** in Kaggle notebook settings for this to work.

In [ ]:
import os; os.chdir('/kaggle/working')

In [ ]:
import os, shutil

GITHUB_REPO = 'https://github.com/Jorshuare/AI-Based-Prediction-for-GNSS-Signal-Degradation.git'
REPO_DIR    = '/kaggle/working/sentinel-gnss'

# Always anchor to /kaggle/working first
%cd /kaggle/working

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repo already cloned — pulling latest ...')
    %cd {REPO_DIR}
    !git pull
else:
    if os.path.exists(REPO_DIR):
        print('Removing stale directory ...')
        shutil.rmtree(REPO_DIR)
    print('Cloning repo ...')
    !git clone {GITHUB_REPO} {REPO_DIR}
    %cd {REPO_DIR}

print(f'\nWorking directory: {os.getcwd()}')
!git log --oneline -5

# Confirm labelled CSV came down with the repo
csv_path = f'{REPO_DIR}/data/labelled/sentinel_gnss_labelled.csv'
assert os.path.exists(csv_path), f'CSV not found at {csv_path}'
import pandas as pd
df = pd.read_csv(csv_path)
print(f'\nDataset: {len(df):,} rows × {len(df.columns)} columns — ready.')

## Step 2 — Install extra dependencies + verify GPU
Kaggle already ships with PyTorch, NumPy, pandas, scikit-learn, matplotlib, seaborn, scipy.  
Only `imbalanced-learn` (SMOTE) and `xgboost` need installing.

In [ ]:
!pip install -q imbalanced-learn xgboost

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {props.total_memory / 1e9:.1f} GB')
    if torch.cuda.device_count() > 1:
        print(f'GPUs     : {torch.cuda.device_count()} (T4 x2 detected — training uses device 0)')
else:
    raise RuntimeError(
        'NO GPU DETECTED.  Go to the notebook Settings panel (right sidebar) '
        '▸ Accelerator ▸ GPU T4 x2 (or P100), then Save and re-run from Step 1.  '
        'Training on CPU takes ~2 h and produces significantly worse results '
        '(no AMP, different gradient dynamics).')

## Step 3 — Set up output directories
On Kaggle, outputs are written to `/kaggle/working/` — no Drive mount required.  
Checkpoints and figures are created inside the cloned repo and are available  
for download from the **Output** tab after the session ends.

In [ ]:
import os

REPO_DIR = '/kaggle/working/sentinel-gnss'

OUTPUT_DIRS = [
    f'{REPO_DIR}/results/models/checkpoints',
    f'{REPO_DIR}/results/figures',
    f'{REPO_DIR}/results/metrics',
]

for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓  {d}')

# Check for any pre-existing checkpoints (e.g. from a partial run in this session)
ckpt_dir = f'{REPO_DIR}/results/models/checkpoints'
ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
if ckpts:
    print(f'\nExisting checkpoints ({len(ckpts)}): {", ".join(ckpts)}')
    print('Add --resume to Step 6 command to continue from the last checkpoint.')
else:
    print('\nNo existing checkpoints — will start fresh.')

## Step 4 — Process new datasets (Deep + Harsh) — Run 12 only

> **In the standard workflow this step is SKIPPED.**  
> The processed feature CSVs (`urbannav_deep_features.csv`, `urbannav_harsh_features.csv`)  
> are generated **locally** and committed to the GitHub repo along with every other dataset.  
> Step 1 (git clone/pull) already brings them in — you do **not** need to upload raw data.
>
> This step only activates if the CSVs are missing — e.g. the local processing step  
> was never done. In that case the cell skips gracefully and the combine step still works  
> with the existing sources.

| Dataset | Expected rows | Expected DEGRADED% |
|---------|---------------|-----------------|
| HK-Deep-Urban-1  (Whampoa, 10 receivers)  | ~14,000 | ~25–35% |
| HK-Harsh-Urban-1 (Mong Kok, 10 receivers) | ~30,000 | ~35–45% |

In [ ]:
%cd /kaggle/working/sentinel-gnss
# ──────────────────────────────────────────────────────────────────────────────
# STANDARD WORKFLOW: Skip this cell.
#   The feature CSVs for Deep and Harsh are generated locally and committed to
#   GitHub along with all other processed datasets.  Step 1 (git clone/pull)
#   already brings them into this session.  Jump straight to Step 5.
#
# FALLBACK (only if CSVs are missing from the repo):
#   Upload raw urbanNav_Deep / urbanNav_Harsh data as a Kaggle dataset, then
#   add it as an input to this notebook and update the paths below.
# ──────────────────────────────────────────────────────────────────────────────

import os, pandas as pd

deep_csv  = 'data/processed/urbannav/urbannav_deep_features.csv'
harsh_csv = 'data/processed/urbannav/urbannav_harsh_features.csv'

if os.path.exists(deep_csv) and os.path.exists(harsh_csv):
    df_deep  = pd.read_csv(deep_csv)
    df_harsh = pd.read_csv(harsh_csv)
    print(f"Deep  CSV already present: {len(df_deep):,} rows")
    print(f"  labels: {df_deep['label'].value_counts().to_dict()}")
    print(f"Harsh CSV already present: {len(df_harsh):,} rows")
    print(f"  labels: {df_harsh['label'].value_counts().to_dict()}")
    print("\nNo processing needed — CSVs came from GitHub. Proceed to Step 5.")
else:
    # Fallback: raw data must be attached as a Kaggle input dataset
    # Update these paths to match your Kaggle dataset mount points:
    deep_dir  = '/kaggle/input/urbannav-deep/urbanNav_Deep'   # <-- update if needed
    harsh_dir = '/kaggle/input/urbannav-harsh/urbanNav_Harsh' # <-- update if needed

    if os.path.exists(deep_dir):
        print("Processing HK-Deep-Urban-1 ...")
        !python src/processing/process_all_datasets.py --source urbannav_deep
    else:
        print(f"[SKIP] {deep_dir} not found — attach the dataset as a Kaggle input")

    if os.path.exists(harsh_dir):
        print("\nProcessing HK-Harsh-Urban-1 ...")
        !python src/processing/process_all_datasets.py --source urbannav_harsh
    else:
        print(f"[SKIP] {harsh_dir} not found — attach the dataset as a Kaggle input")

    print("\nRebuilding combined sentinel_gnss_labelled.csv ...")
    !python src/processing/process_all_datasets.py --combine

# Verify combined dataset
df = pd.read_csv('data/labelled/sentinel_gnss_labelled.csv')
print(f"\nCombined dataset: {len(df):,} rows × {len(df.columns)} columns")
by_src = df['source'].value_counts()
deep_srcs  = by_src[by_src.index.str.startswith('urbannav_deep')].sum()
harsh_srcs = by_src[by_src.index.str.startswith('urbannav_harsh')].sum()
print(f"  urbannav_deep_*  : {deep_srcs:,} rows")
print(f"  urbannav_harsh_* : {harsh_srcs:,} rows")

## Step 5 — Build feature windows
Reads `data/labelled/sentinel_gnss_labelled.csv` → produces sliding-window tensors.  
**Use `--force` after adding Deep + Harsh (Step 4) to regenerate windows with new data.**  
Skips automatically if windows already exist and `--force` is not passed.

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Run 12 change: --force is required after adding Deep + Harsh in Step 4.
# Without --force, feature_prep skips if windows already exist.
#
# Exclusions in effect via DEFAULT_EXCLUDE_SOURCES (see feature_prep.py):
#   Drones         : UAV open-sky, non-vehicular, 100% CLEAN — no degradation signal.
#   Oxford         : 2014 GPS-only hardware; labels from position-sigma, not C/N₀.
#                    96% DEGRADED/WARNING — inflated DEGRADED class, wrong mechanism.
#   NCLT           : 2012 GPS module; num_satellites always 0 (logging bug) →
#                    DOP proxy values are noise; no C/N₀.
#   Tokyo Shinjuku : 31,265 rows, 92-94% CLEAN, DOP imputed from satellite count.
#                    Was 51% of entire dataset — dominated CLEAN class, suppressed
#                    WARNING recall.  Reserved as cross-city evaluation set.
#   Tokyo Odaiba u-blox: 100% CLEAN, DOP imputed.  Same issue.
#   tokyo_odaiba_trimble stays in VAL (12,398 rows, 97.7% CLEAN) to provide
#   CLEAN examples for the balanced early-stopping val subset.
#
# Run 12 new sources (after Step 4):
#   urbannav_deep_*  : 10 receivers (HK Whampoa) — consumer phones → train
#                      Professional NovAtel F9P → val (cross-receiver eval)
#   urbannav_harsh_* : 10 receivers (HK Mong Kok) — all non-professional → train
#                      NovAtel FlexPak6 → val (cross-receiver eval)

# Standard SMOTE windows (used by baselines, ablations, evaluate)
!python -m src.models.feature_prep --force

# No-SMOTE windows (used by deep neural net to avoid SMOTE-induced time-coherence issues)
!python -m src.models.feature_prep --no_smote --force

# Sanity check — Run 12 expected after adding Deep + Harsh:
#   DEGRADED (class 2) in train should be >> 555 (Run 10/11 value)
import numpy as np

for tag, wdir in [('SMOTE', 'windows'), ('no-SMOTE', 'windows_no_smote')]:
    print(f'\nWindow shapes ({tag}):')
    for split in ('train', 'val', 'test'):
        d = np.load(f'data/processed/{wdir}/{split}.npz')
        y0_tag = 'y_0s present' if 'y_0s' in d else 'y_0s MISSING'
        c = int(np.sum(d['y_5s'] == 0))
        w = int(np.sum(d['y_5s'] == 1))
        g = int(np.sum(d['y_5s'] == 2))
        print(f'  {split:5s}  X={d["X"].shape}  '
              f'CLEAN={c:,}  WARNING={w:,}  DEGRADED={g:,}  [{y0_tag}]')

## Step 6 — Train
Checkpoints are saved to `results/models/checkpoints/` inside the working directory.  
Run `--resume` to continue from the last checkpoint within the same session.  
Add `--resume` to the command below if you are re-running after an interruption.

In [ ]:
import glob, os

REPO_DIR  = '/kaggle/working/sentinel-gnss'
ckpt_dir  = f'{REPO_DIR}/results/models/checkpoints'

# ── Optional: uncomment to wipe existing checkpoints and start fresh ──
# for f in glob.glob(ckpt_dir + '/*.pt'):
#     os.remove(f)
#     print(f'Removed: {f}')

ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
if ckpts:
    print(f'Existing checkpoints ({len(ckpts)}):')
    for c in ckpts:
        print(f'  {c}')
    print('\nTo resume: add --resume to the training command below.')
else:
    print('No existing checkpoints — starting fresh.')

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Trained WITHOUT SMOTE (SMOTE creates temporally incoherent synthetic windows).
#
# ══════════════════════════════════════════════════════════════════
#  RUN 12 — Add HK-Deep-Urban-1 + HK-Harsh-Urban-1 to training
# ══════════════════════════════════════════════════════════════════
# Run 10/11 bottleneck: DEGRADED F1 at +5s = 0.274.
#   Only 555 natural DEGRADED training rows before SMOTE.
#   Only 55 DEGRADED test windows (4.3% of test set).
#
# Run 12 change: Deep + Harsh add ~44,000 new rows; DEGRADED training
#   rows expected to grow from ~555 → ~5,000+ before SMOTE.
#   Expected DEGRADED F1 at +5s: 0.40–0.55.
#
# Architecture: TransformerEncoder (2L, 8H, d=128, d_ff=512) →
#               BiLSTM (2L, hidden=256) → 3 output heads + aux head
# Loss: focal_gamma=1.0, class_weights=[1.0, 2.0, 5.0], smoothing=0.1
# Training: AdamW, patience=50, min_epoch_for_best=15, batch_size=256
# ══════════════════════════════════════════════════════════════════
!python -m src.models.train --batch_size 256 --window_dir data/processed/windows_no_smote

## Step 7 — Evaluate
Loads `checkpoint_best.pt`, runs all 14 analyses, saves figures to `results/figures/`.

In [ ]:
%cd /kaggle/working/sentinel-gnss
# --tune_thresholds: sweeps WARNING/DEGRADED probability thresholds on val set
#   to maximise macro-F1. Free improvement — no retraining needed.
# --temperature_scaling: finds optimal calibration temperature T on val set
#   (Guo et al. 2017, ICML) before threshold tuning.
#   T > 1 reduces over-confidence in DEGRADED predictions for borderline
#   WARNING windows — targets the W→DEG false alarm problem.
# --window_dir: val/test splits are identical to SMOTE dir.
!python -m src.models.evaluate \
    --tune_thresholds \
    --temperature_scaling \
    --window_dir data/processed/windows_no_smote

## Step 8 — View all figures inline

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob('/kaggle/working/sentinel-gnss/results/figures/*.png'))
print(f'{len(figs)} figures saved (also available in the Output tab):')
for f in figs:
    name = f.split('/')[-1]
    print(f'  {name}')
    display(Image(filename=f, width=950))

## Step 9 — Baselines (Tier 1–3)

Run trivial, domain-rule, and classical-ML baselines for the comparison table.  
Takes ~5–10 min on CPU (RF 500 trees, XGBoost 400 estimators).  
XGBoost was already installed in Step 2.

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Run all Tier 1–3 baselines (MajorityClass, CNR threshold, RF, XGBoost)
!python -m src.models.baselines

## Step 10 — Ablations (Tier 4)

Train LSTM-only and Transformer-only variants.  
Each uses its own checkpoint directory (`checkpoints_lstm_only/`, `checkpoints_transformer_only/`).  
Same data, same loss, same hyperparameters — only architecture differs.

In [ ]:
%cd /kaggle/working/sentinel-gnss
# LSTM-only ablation — no Transformer encoder
# Expected: fewer epochs to converge, lower macro-F1 than full model
!python -m src.models.train --model_type lstm_only --batch_size 256

# Evaluate LSTM-only on test set → saves metrics_test_lstm_only.json
# Required before baselines.py --include_ablations can show correct numbers
!python -m src.models.evaluate \
    --model_type lstm_only \
    --tune_thresholds \
    --temperature_scaling \
    --window_dir data/processed/windows_no_smote

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Transformer-only ablation — no LSTM (mean pooling over sequence)
# Expected: lower macro-F1 than full model due to missing sequential state
!python -m src.models.train --model_type transformer_only --batch_size 256

# Evaluate Transformer-only on test set → saves metrics_test_transformer_only.json
!python -m src.models.evaluate \
    --model_type transformer_only \
    --tune_thresholds \
    --temperature_scaling \
    --window_dir data/processed/windows_no_smote

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Print the full comparison table (includes DL ablation results if available)
!python -m src.models.baselines --include_ablations

## Output

All results are written to `/kaggle/working/sentinel-gnss/results/`:

```
results/
├── models/
│   └── checkpoints/          ← .pt checkpoint files
├── figures/                  ← all evaluation plots (.png)
└── metrics/                  ← JSON metric files per model
```

Download them from the **Output** tab in the Kaggle notebook UI, or add this notebook  
as a data source in another notebook to reference the saved files directly.